In [8]:
from Bio import Entrez
import xml.etree.ElementTree as ET
import pandas as pd
import os
from geopy.geocoders import Nominatim
import folium
import seaborn as sns

species = "Loxodonta_africana"

# For running it locally
if os.getcwd().startswith("/home/lakrids"):
    path_prefix = "/home/lakrids/GenomeDK"
else:
    path_prefix = "/faststorage/project"

Entrez.email = "laurids@bio.au.dk"

In [ ]:
pca_df = pd.read_csv(f"{path_prefix}/megaFauna/sa_megafauna/results/{species}/PCA/pca_dataset_{species}.txt", sep='\t')
pca_df

,IND_ID,PC1,PC2,ecotype,biomaterial_provider,geo_loc_name
0,SAMEA115942035,-37.733982,35.792614,NaN,NaN,NaN
1,SAMEA115942036,-38.044960,35.723175,NaN,NaN,NaN
2,SAMEA115942037,-37.119762,34.429676,NaN,NaN,NaN
3,SAMEA115942038,-37.944050,36.168140,NaN,NaN,NaN
4,SAMEA115942039,-38.275757,36.330074,NaN,NaN,NaN
...,...,...,...,...,...,...
161,SAMN28571074,-35.032623,30.190683,NaN,"Indianapolis Zoological Society, Inc.",Uganda
162,SAMN28571075,-36.945763,34.027195,NaN,"Indianapolis Zoological Society, Inc.",Mozambique
163,SAMN28571081,-37.582542,35.299534,NaN,Utah's Hogle Zoo,NaN
164,SAMN28571082,-37.996320,36.274597,NaN,Utah's Hogle Zoo,South Africa: Kruger


In [23]:
import pandas as pd

# Read the Excel file sheets
data_10 = pd.read_excel('/home/lakrids/Documents/UNI/Thesis/41467_2026_71262_MOESM3_ESM.xlsx', sheet_name='Supplementary Data 10', skiprows=1)
data_1 = pd.read_excel('/home/lakrids/Documents/UNI/Thesis/41467_2026_71262_MOESM3_ESM.xlsx', sheet_name='Supplementary Data 1', skiprows=1)

# Clean up column names (remove extra spaces)
data_10.columns = data_10.columns.str.strip()
data_1.columns = data_1.columns.str.strip()

# Step 1: Merge Data 10 with Data 1 to link NCBI IDs to geography
# From Data 10, we need: Sample accession (NCBI ID) and Sample (DNA sample)
# From Data 1, we need: DNA sample, Country, and Location

# First, merge the two supplementary datasets
metadata = data_10.merge(
    data_1[['DNA sample', 'Country', 'Location']], 
    left_on='Sample',  # Column in Data 10
    right_on='DNA sample',  # Column in Data 1
    how='left'
)

# Step 2: Keep only unique NCBI sample accessions with their geography
# (since Data 10 has multiple run accessions per sample accession)
metadata_unique = metadata[['Sample accession', 'Country', 'Location']].drop_duplicates()

# Step 3: Merge with your pca_df
# Assuming your pca_df has a column called 'sample_ID' with NCBI sample accessions
pca_df_with_geo = pca_df.merge(
    metadata_unique,
    left_on='IND_ID',  # Adjust this to your actual column name
    right_on='Sample accession',
    how='left'
)

# Drop the duplicate Sample accession column if you want
pca_df_with_geo = pca_df_with_geo.drop('Sample accession', axis=1)

# Step 4: Fill Country and Location with geo_loc_name when they're NaN
# Only fill when BOTH Country and Location are NaN and geo_loc_name has a value
mask = (pca_df_with_geo['Country'].isna() & 
        pca_df_with_geo['Location'].isna() & 
        pca_df_with_geo['geo_loc_name'].notna())

pca_df_with_geo.loc[mask, 'Country'] = pca_df_with_geo.loc[mask, 'geo_loc_name']
pca_df_with_geo.loc[mask, 'Location'] = pca_df_with_geo.loc[mask, 'geo_loc_name']

# Check the results
print(f"Original PCA samples: {len(pca_df)}")
print(f"Samples with geography data: {pca_df_with_geo['Country'].notna().sum()}")
print(f"Samples without geography data: {pca_df_with_geo['Country'].isna().sum()}")
print(f"Samples filled from geo_loc_name: {mask.sum()}")

# View samples with geography
print("\nSamples with geography:")
print(pca_df_with_geo[pca_df_with_geo['Country'].notna()][['IND_ID', 'Country', 'Location']])

Original PCA samples: 166
Samples with geography data: 152
Samples without geography data: 14
Samples filled from geo_loc_name: 9

Samples with geography:
             IND_ID               Country              Location
0    SAMEA115942035                 Kenya              Amboseli
1    SAMEA115942036                 Kenya              Amboseli
2    SAMEA115942037                 Kenya              Amboseli
3    SAMEA115942038                 Kenya              Amboseli
4    SAMEA115942039                 Kenya              Amboseli
..              ...                   ...                   ...
147    SAMN03447174               missing               missing
161    SAMN28571074                Uganda                Uganda
162    SAMN28571075            Mozambique            Mozambique
164    SAMN28571082  South Africa: Kruger  South Africa: Kruger
165    SAMN28571084              Zimbabwe              Zimbabwe

[152 rows x 3 columns]


In [ ]:
pca_df_with_geo.drop(columns=["ecotype", "biomaterial_provider"])


,IND_ID,PC1,PC2,geo_loc_name,Country,Location
0,SAMEA115942035,-37.733982,35.792614,NaN,Kenya,Amboseli
1,SAMEA115942036,-38.044960,35.723175,NaN,Kenya,Amboseli
2,SAMEA115942037,-37.119762,34.429676,NaN,Kenya,Amboseli
3,SAMEA115942038,-37.944050,36.168140,NaN,Kenya,Amboseli
4,SAMEA115942039,-38.275757,36.330074,NaN,Kenya,Amboseli
...,...,...,...,...,...,...
161,SAMN28571074,-35.032623,30.190683,Uganda,Uganda,Uganda
162,SAMN28571075,-36.945763,34.027195,Mozambique,Mozambique,Mozambique
163,SAMN28571081,-37.582542,35.299534,NaN,NaN,NaN
164,SAMN28571082,-37.996320,36.274597,South Africa: Kruger,South Africa: Kruger,South Africa: Kruger


In [4]:
# Fetching metadata from the NCBI database
# Have a look at the added columns or go to the sample pages to decide how to make the geography column.
def fetch_selected_attributes(sample_id, target_attrs):
    results = {attr: None for attr in target_attrs}

    try:
        # Fetch BioSample XML
        handle = Entrez.efetch(db="biosample", id=sample_id, rettype="xml")
        xml_data = handle.read()
        root = ET.fromstring(xml_data)

        # Extract target attributes
        for attr_elem in root.findall(".//Attribute"):
            name = attr_elem.attrib.get("harmonized_name")
            if name in target_attrs:
                results[name] = attr_elem.text

    except Exception as e:
        print(f"Error fetching {sample_id}: {e}")

    return results

In [7]:
# Define attributes to extract
target_attrs = {"ecotype", "biomaterial_provider", "geo_loc_name"}

# Apply to your dataframe
pca_df[list(target_attrs)] = pca_df["IND_ID"].apply(
    lambda sid: pd.Series(fetch_selected_attributes(sid, target_attrs))
)
pca_df

,IND_ID,PC1,PC2,ecotype,biomaterial_provider,geo_loc_name
0,SAMN13242356,-3999.4644,137.03305,None,None,None
1,SAMN13242357,-3886.8855,134.26918,None,None,None
2,SAMN13242358,-3869.9710,135.25449,None,None,None
3,SAMN13242359,-3929.3960,134.71518,None,None,None
4,SAMN13242360,-3936.5889,134.55568,None,None,None
5,SAMN13242366,4118.8890,-4457.16500,None,None,None
6,SAMN13247121,-3782.7827,143.80104,None,None,None
7,SAMN13247122,5020.6284,8630.95600,None,None,None
8,SAMN13247123,4384.3125,3406.29320,None,None,None
9,SAMN13247124,4646.2666,-4611.43950,None,None,None


In [8]:
# This is probably very specific for the tigers
def resolve_geography(row):
    biomaterial = row.get("biomaterial_provider")
    ecotype = row.get("ecotype")

    if biomaterial and biomaterial.lower() != "wild":
        return biomaterial
    elif ecotype:
        return ecotype
    else:
        return None

In [9]:
pca_df["geography"] = pca_df.apply(resolve_geography, axis=1)

# Some manual editing of the geographies
pca_df.loc[pca_df["IND_ID"] == "SAMD00283119", "geography"] = "Preservation and Research Center, Yokohama"


# Removing the two columns that we used to assemble the "geography" column
pca_df = pca_df.drop('ecotype', axis=1)
pca_df = pca_df.drop('biomaterial_provider', axis=1)

pca_df['geography_simple'] = pca_df['geography']#.str.split(',').str[0]



In [11]:
manual_locations = {
    "San Deago Zoo": (32.73620674440929, -117.15096344662129),
    "Preservation and Research Center, Yokohama": (35.47066373451937, 139.62552761715486)
}

def geocode_location(name):
    geolocator = Nominatim(user_agent="geo_lookup")
    if name in manual_locations:
        return pd.Series(manual_locations[name])
    try:
        location = geolocator.geocode(name)
        if location:
            return pd.Series([location.latitude, location.longitude])
    except:
        pass
    return pd.Series([None, None])

pca_df[['latitude', 'longitude']] = pca_df['geography'].apply(geocode_location)
pca_df.head()

,IND_ID,PC1,PC2,geo_loc_name,geography,geography_simple,latitude,longitude
0,SAMN13242356,-3999.4644,137.03305,None,None,None,44.933143,7.540121
1,SAMN13242357,-3886.8855,134.26918,None,None,None,44.933143,7.540121
2,SAMN13242358,-3869.9710,135.25449,None,None,None,44.933143,7.540121
3,SAMN13242359,-3929.3960,134.71518,None,None,None,44.933143,7.540121
4,SAMN13242360,-3936.5889,134.55568,None,None,None,44.933143,7.540121


In [12]:
# Drop rows without coordinates
df = pca_df.dropna(subset=['latitude', 'longitude'])

#df_el = pd.read_csv(f"/home/{user}/megaFauna/sa_megafauna/data/Elephas_maximus/PCA/pca_results.txt")

# Create color map for each unique location
locations = df['geography_simple'].fillna("Unknown")
unique_locations = locations.unique()
palette = sns.color_palette("husl", len(unique_locations)).as_hex()
location_color_map = dict(zip(unique_locations, palette))

# Count samples per location
location_counts = locations.value_counts()

# Create base map centered on average location
map_center = [df['latitude'].mean(), df['longitude'].mean()]
sample_map = folium.Map(location=map_center, zoom_start=5)

# Add sample markers
for _, row in df.iterrows():
    location_name = row['geography_simple']
    color = location_color_map[location_name]
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=15,
        popup=f"{row['IND_ID']} ({location_name})",
        color=color,
        fill=True,
        fill_opacity=0.8
    ).add_to(sample_map)

# Add legend (as a custom HTML element)
legend_html = """
<div style="position: fixed; 
     bottom: 30px; left: 30px; width: 250px; height: auto; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;
     padding: 10px;">
     <b>Sample Locations</b><br>
"""
for loc in unique_locations:
    legend_html += f"""<i style="background:{location_color_map[loc]};width:10px;height:10px;display:inline-block;margin-right:5px;"></i>
                       {loc} ({location_counts[loc]})<br>"""
legend_html += "</div>"

sample_map.get_root().html.add_child(folium.Element(legend_html))

# Save or display
sample_map.save(f"{path_prefix}/megaFauna/sa_megafauna/results/{species}/PCA/sample_locations_map_{species}.html")

KeyError: None

In [26]:
pca_df_with_geo.to_csv(f"{path_prefix}/megaFauna/sa_megafauna/results/{species}/PCA/pca_dataset_{species}.txt", sep='\t', index=False)